# D1b 손실·역전파·옵티마이저 — 실습 (W2)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. MSE·교차 엔트로피를 **손 계산과 대조**해 검산한다
2. autograd로 기울기를 검산하고, **경사하강 루프를 손으로** 돌린다
3. **학습률의 세 가지 운명**(수렴/진동/발산)을 그래프로 확인한다
4. **1학기 M4 선형회귀를 밑바닥부터 재현**하고 옵티마이저로 교체한다
5. **XOR 가중치를 Adam으로 학습**시켜 지난주 손 가중치와 비교한다

**7단계 멘탈모델 초점:** 손실 + 최적화

## Part A. 활성화 함수 — 곡선 직접 그리기
은닉층=ReLU 기본, Sigmoid는 이진 분류 출력(M5), Tanh는 RNN(D3)에서 재등장.

In [ ]:
import torch                                            # PyTorch
import torch.nn as nn                                   # 손실 함수(Part B)
import matplotlib.pyplot as plt                         # 그래프

xs = torch.linspace(-5, 5, 200)                         # -5~5 구간 200점
plt.figure(figsize=(6, 4))                              # 크기
plt.plot(xs, torch.relu(xs), label='ReLU')              # max(0,x)
plt.plot(xs, torch.___(xs), label='Sigmoid')            # ✍️ 빈칸: 0~1 S자 함수
plt.plot(xs, torch.tanh(xs), label='Tanh')              # -1~1 S자
plt.axhline(0, color='gray', lw=0.5)                    # 기준선
plt.axvline(0, color='gray', lw=0.5)                    # 기준선
plt.title('Activation functions')                       # 제목(영어)
plt.legend(); plt.grid(True); plt.show()                # Sigmoid가 양끝에서 평평해짐 = 기울기 소실의 씨앗

## Part B. 손실 두 가지 — 손 계산과 대조 ⭐
설명서 §3(MSE=0.625)·§4(CE≈0.17)의 손 계산을 PyTorch가 재현하는지 검산합니다.

In [ ]:
pred = torch.tensor([2.5, 0.0])                         # 예측
target = torch.tensor([3.0, -1.0])                      # 정답
mse_manual = ((pred - target) ** 2).___()               # ✍️ 빈칸: 제곱오차의 평균
mse_torch = nn.MSELoss()(pred, target)                  # PyTorch MSE
print('MSE 손계산:', mse_manual.item(), '| torch:', mse_torch.item())  # 둘 다 0.625

logits = torch.tensor([[2.0, 0.0, -1.0]])               # 모델 출력(logit) — softmax 전!
label = torch.tensor([0])                               # 정답 = 클래스 0
probs = torch.softmax(logits, dim=___)                  # ✍️ 빈칸: 클래스 차원(1번)으로 확률화
print('softmax 확률:', probs.round(decimals=3))         # (0.844, 0.114, 0.042)
ce_manual = -torch.log(probs[0, 0])                     # -log(정답 확률)
ce_torch = nn.CrossEntropyLoss()(logits, label)         # PyTorch CE (softmax 포함!)
print('CE 손계산:', round(ce_manual.item(), 4), '| torch:', round(ce_torch.item(), 4))  # 둘 다 ~0.1698

bad_logits = torch.tensor([[-1.0, 0.0, 2.0]])           # 자신 있게 틀린 경우
print('크게 틀림 CE:', round(nn.CrossEntropyLoss()(bad_logits, label).item(), 4))  # ~3.17 — 벌점 폭발

## Part C. autograd 검산
y = w² + 2w를 w=3에서 미분: 손으로는 2w+2 = 8. autograd도 같은 답을 내는지.

In [ ]:
w = torch.tensor(3.0, requires_grad=True)               # 미분 대상 변수
y = w**2 + 2*w                                          # 연산이 기록됨
y.___()                                                 # ✍️ 빈칸: 역전파 실행 메서드
print('autograd dy/dw =', w.grad.item())                # 8.0 — 손 미분(2·3+2)과 일치

## Part D. 경사하강을 손으로 — backward → 갱신 → zero ⭐
f(w) = (w−3)²의 골짜기를 30걸음에 내려갑니다. **수동 갱신은 `no_grad` 안에서, `.grad`는 매번 0으로.**

In [ ]:
w = torch.tensor(0.0, requires_grad=True)               # 시작점 w=0
LR = 0.1                                                # 학습률(보폭)
for step in range(30):
    loss = (w - 3) ** 2                                 # 손실
    loss.backward()                                     # 기울기 계산(역전파)
    with torch.no_grad():                               # 갱신은 기록 없이
        w -= LR * ___                                   # ✍️ 빈칸: 기울기가 담긴 속성
    w.grad.zero_()                                      # 기울기 초기화(누적 방지!)
print('30걸음 후 w =', round(w.item(), 4))              # ~2.9963 — 골짜기(3) 도착

In [ ]:
def gd_path(lr, steps=15):                              # lr별 경사하강 궤적
    w = torch.tensor(0.0, requires_grad=True)           # 항상 w=0에서 출발
    hist = [0.0]                                        # 궤적 기록
    for _ in range(steps):
        loss = (w - 3) ** 2                             # 같은 골짜기
        loss.backward()                                 # 기울기
        with torch.no_grad():
            w -= lr * w.grad                            # 한 걸음
        w.grad.zero_()                                  # 초기화
        hist.append(w.item())                           # 기록
    return hist

plt.figure(figsize=(6, 4))                              # 세 가지 운명 그리기
for lr in [0.1, 0.95, 1.05]:
    plt.plot(gd_path(lr), 'o-', label=f'lr={lr}')       # 수렴/진동/발산
plt.axhline(3, color='gray', ls='--', label='target w=3')  # 목표
plt.xlabel('step'); plt.ylabel('w')                     # 축(영어)
plt.title('Three fates of learning rate')               # 제목(영어)
plt.legend(); plt.grid(True); plt.show()                # lr이 운명을 가른다

## Part E. 1학기 M4 선형회귀, 밑바닥부터 ⭐⭐
데이터: y = 2x + 3 + 잡음. `LinearRegression().fit()`이 하던 일을 같은 리듬으로 직접 합니다.

In [ ]:
torch.manual_seed(42)                                   # 재현성
x = torch.linspace(-2, 2, 100)                          # 입력 100점
y = 2 * x + 3 + 0.3 * torch.randn(100)                  # 정답: 기울기2·절편3 + 잡음

w = torch.randn(1, requires_grad=True)                  # 기울기(난수 시작)
b = torch.zeros(1, requires_grad=True)                  # 절편(0 시작)
LR = 0.1                                                # 학습률
for epoch in range(200):
    pred = w * x + b                                    # 순전파(모델=직선)
    loss = ((pred - y) ** 2).mean()                     # MSE
    loss.backward()                                     # 역전파
    with torch.no_grad():                               # 기록 없이 갱신
        w -= LR * w.grad                                # w 한 걸음
        b -= LR * ___                                   # ✍️ 빈칸: b의 기울기
    w.grad.zero_(); b.grad.zero_()                      # 초기화
print('학습 결과: w =', round(w.item(), 3), '| b =', round(b.item(), 3))  # ~2.0 / ~3.0

plt.figure(figsize=(6, 4))                              # 결과 그리기
plt.scatter(x, y, s=12, alpha=0.6, label='data')        # 데이터
with torch.no_grad():
    plt.plot(x, w * x + b, 'r-', lw=2, label='learned line')  # 학습된 직선
plt.xlabel('x'); plt.ylabel('y')                        # 축(영어)
plt.title('Linear regression from scratch (M4 revisited)')  # 제목(영어)
plt.legend(); plt.grid(True); plt.show()                # .fit()의 정체가 이 루프

## Part F. 옵티마이저로 교체 — zero_grad → backward → step
수동 갱신 2줄을 옵티마이저에 맡깁니다. 결과는 같고, 코드는 다음 주 학습 루프의 심장이 됩니다.

In [ ]:
torch.manual_seed(42)                                   # 같은 조건으로 재시작
wo = torch.randn(1, requires_grad=True)                 # 기울기
bo = torch.zeros(1, requires_grad=True)                 # 절편
opt = torch.optim.SGD([wo, bo], lr=0.1)                 # 갱신 담당자 고용(SGD)
for epoch in range(200):
    opt.___()                                           # ✍️ 빈칸: ① 기울기 초기화
    loss = ((wo * x + bo - y) ** 2).mean()              # ② 순전파+손실
    loss.backward()                                     # ③ 역전파
    opt.___()                                           # ✍️ 빈칸: ④ 가중치 갱신
print('옵티마이저 결과: w =', round(wo.item(), 3), '| b =', round(bo.item(), 3))  # Part E와 동일 수렴

## Part G. 대단원 — XOR 가중치를 기계가 찾다 ⭐⭐
지난주(D1a)엔 가중치를 손으로 정했습니다. 이번엔 **난수에서 출발해 Adam으로 학습**시킵니다.

> 은닉 뉴런은 **8개로 넉넉하게** 둡니다. 손 설계(D1a)는 2개면 충분했지만, **학습으로 찾을 땐** 최소 구조의 손실 지형이 험해 잘못된 골짜기에 자주 빠집니다(심화에서 직접 확인). 편향도 작은 난수로 시작합니다(모든 뉴런이 꺼진 채 시작하는 것 방지).

In [ ]:
torch.manual_seed(0)                                    # 재현성
X4 = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])  # XOR 입력(D1a와 동일)
target = torch.tensor([[0.], [1.], [1.], [0.]])         # XOR 정답
W1 = torch.randn(2, 8, requires_grad=True)              # 입력2 → 은닉8, 전부 난수 시작!
b1 = (0.5 * torch.randn(8)).requires_grad_()            # 편향도 작은 난수(죽은 ReLU 방지)
w2 = torch.randn(8, 1, requires_grad=True)              # 은닉8 → 출력1
b2 = torch.zeros(1, requires_grad=True)                 # 출력 편향
opt = torch.optim.Adam([W1, b1, w2, b2], lr=0.1)        # Adam 옵티마이저

losses = []                                             # 손실 기록
for step in range(300):
    h = torch.relu(X4 @ W1 + b1)                        # 순전파(D1a 구조 그대로, 뉴런만 8개)
    out = h @ w2 + b2                                   # 출력
    loss = ((out - target) ** 2).mean()                 # MSE
    opt.zero_grad(); loss.backward(); opt.step()        # 3줄 리듬
    losses.append(loss.item())                          # 기록

print('최종 손실:', round(losses[-1], 6))               # ~0
print('학습 후 예측:', out.detach().squeeze().round(decimals=3))  # ~0,1,1,0
print('반올림 예측 :', out.detach().squeeze().round())  # 0,1,1,0 — XOR 해결!
print('학습된 W1 (2x8, 일부):')                         # 손 설계(2x2, 값 1)와 전혀 다름
print(W1.detach()[:, :4].round(decimals=2))             # 앞 4개 뉴런만 표시
print('학습된 w2:', w2.detach().squeeze().round(decimals=2))  # 그래도 정답은 같음

plt.figure(figsize=(6, 4))                              # 손실 곡선
plt.plot(losses)                                        # step별 손실
plt.xlabel('step'); plt.ylabel('MSE loss')              # 축(영어)
plt.title('XOR training loss (Adam)')                   # 제목(영어)
plt.grid(True); plt.show()                              # 손실이 0 근처로

> **좋은 가중치는 유일하지 않다.** 학습된 W1은 지난주 손 가중치와 다르지만 XOR을 똑같이 풉니다. 손실 산맥에는 골짜기가 여러 개 — 초기 난수(시드)가 다르면 다른 골짜기에 내려앉습니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "logit이 (1, 1, 1)이면 softmax와 CE가 어떻게 되는지 내 계산을 채점해 줘."
- "lr=0.5로 (w−3)²를 내려가면 몇 걸음에 수렴하는지 내가 예측할 테니 검산해 줘."
- "`backward()`와 `step()`의 역할 차이를 내가 설명할 테니 틀린 곳을 질문으로 짚어 줘."
- "XOR 학습에서 시드를 바꾸면 왜 다른 가중치가 나오는지 손실 지형으로 설명해 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. MSE(0.625)·CE(0.17)를 손 계산과 대조해 검산했다
2. autograd(8.0 검산) + 경사하강 수동 루프로 학습의 리듬(backward→갱신→zero)을 익혔고, lr의 세 운명을 그렸다
3. M4 선형회귀를 밑바닥 재현(w→2, b→3)하고 옵티마이저로 교체, XOR 가중치를 Adam으로 찾았다

**스스로 점검**
- [ ] CE에 softmax를 또 넣으면 안 되는 이유를 안다
- [ ] lr 0.1/0.95/1.05 곡선의 모양 차이를 설명할 수 있다
- [ ] `.grad`를 매번 0으로 만드는 이유(누적)를 안다
- [ ] `backward()`(계산)와 `step()`(갱신)의 역할을 구분한다
- [ ] "좋은 가중치는 유일하지 않다"를 XOR 실험으로 설명할 수 있다

**🔹심화 (선택)**
- **최소 구조의 험한 지형 체험:** Part G에서 은닉을 8 → **2**로 줄여(W1을 `(2,2)`, b1·w2도 맞춰) 시드 0~5로 돌려 보세요. 대부분 손실이 0.25나 0.17에서 멈춥니다(잘못된 골짜기). 손 설계(D1a)로는 2개면 충분했는데 **학습으로 찾을 땐 뉴런 여유가 지형을 부드럽게** 만든다 — 딥러닝이 실제로 "필요보다 큰 모델"을 쓰는 이유의 맛보기.
- Part G의 시드를 1, 2로 바꿔 보세요 — 은닉 8개여도 가끔 실패합니다. 학습된 가중치가 시드마다 다른데 정답은 같은 것도 확인.
- Part F의 SGD를 `torch.optim.Adam`(lr=0.1)으로 바꿔 수렴 속도를 비교해 보세요.